# Imports

In [0]:
from pyspark.sql.functions import trim, when, length, lit, col, row_number, lower as lower_spark, concat_ws, coalesce, current_timestamp, sha2
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
from pyspark.sql.functions import sum as sum_spark, lower as lower_spark, upper as upper_spark, countDistinct, first

In [0]:
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

SILVER_DRIVERS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.drivers"
RACE_RESULTS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.race_results"
QUALIFYING_RESULTS_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.qualifying_results"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

In [0]:
def clean_string(column_name: str):
    value = trim(col(column_name))

    return (
        when(length(value) == 0, lit(None).cast("string"))
         .otherwise(value)
    )

# Drivers

In [0]:
def extract_driver_candidates(table_name: str, source_priority: int):
    return (
        spark.table(table_name)
        .select(
            col("season").cast("int").alias("season"),
            col("round").cast("int").alias("round"),

            lower_spark(
                clean_string("DriverId")
            ).alias("driver_id"),

            clean_string("DriverNumber").alias("driver_number"),

            upper_spark(
                clean_string("Abbreviation")
            ).alias("abbreviation"),

            clean_string("BroadcastName").alias("broadcast_name"),
            clean_string("FirstName").alias("first_name"),
            clean_string("LastName").alias("last_name"),
            clean_string("FullName").alias("full_name"),

            upper_spark(
                clean_string("CountryCode")
            ).alias("country_code"),

            clean_string("HeadshotUrl").alias("headshot_url"),

            col("_source_entity").alias("source_entity"),
            col("_source_file").alias("source_file"),

            col("_source_file_modification_time")
                .alias("source_modified_at"),

            col("_ingested_at")
                .alias("bronze_ingested_at"),

            col("_rescued_data"),

            lit(source_priority).alias("_source_priority")
        )
    )

In [0]:
race_drivers_df = extract_driver_candidates(
    RACE_RESULTS_TABLE,
    source_priority=2
)

qualifying_drivers_df = extract_driver_candidates(
    QUALIFYING_RESULTS_TABLE,
    source_priority=1
)

driver_candidates_df = race_drivers_df.unionByName(
    qualifying_drivers_df
)

In [0]:
drivers_columns = [
    "driver_id",
    "driver_number",
    "abbreviation",
    "broadcast_name",
    "first_name",
    "last_name",
    "full_name",
    "country_code",
    "headshot_url"
]

hash_expression = concat_ws(
    "||",
    *[
        coalesce(
            col(column).cast("string"),
            lit("<NULL>")
        )
        for column in drivers_columns
    ]
)

driver_candidates_df = (
    driver_candidates_df
    .withColumn(
        "record_hash",
        sha2(hash_expression, 256)
    )
    .withColumn(
        "silver_updated_at",
        current_timestamp()
    )
)

## Validaciones Driver

In [0]:
ordering = (
    Window
    .partitionBy("driver_id")
    .orderBy(
        col("season").desc(),
        col("round").desc(),
        col("_source_priority").desc(),
        col("source_modified_at").desc_nulls_last(),
        col("bronze_ingested_at").desc_nulls_last()
    )
)

full_window = ordering.rowsBetween(
    Window.unboundedPreceding,
    Window.unboundedFollowing
)

driver_candidates_df = (
    driver_candidates_df

    .withColumn(
        "canonical_driver_number",
        first(
            "driver_number",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "canonical_abbreviation",
        first(
            "abbreviation",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "canonical_broadcast_name",
        first(
            "broadcast_name",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "canonical_first_name",
        first(
            "first_name",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "canonical_last_name",
        first(
            "last_name",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "canonical_full_name",
        first(
            "full_name",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "canonical_country_code",
        first(
            "country_code",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "canonical_headshot_url",
        first(
            "headshot_url",
            ignorenulls=True
        ).over(full_window)
    )

    .withColumn(
        "_row_number",
        row_number().over(ordering)
    )

    # Una única fila física por driver_id
    .filter(col("_row_number") == 1)

    .select(
        "driver_id",

        col("canonical_driver_number")
            .alias("driver_number"),

        col("canonical_abbreviation")
            .alias("abbreviation"),

        col("canonical_broadcast_name")
            .alias("broadcast_name"),

        col("canonical_first_name")
            .alias("first_name"),

        col("canonical_last_name")
            .alias("last_name"),

        col("canonical_full_name")
            .alias("full_name"),

        col("canonical_country_code")
            .alias("country_code"),

        col("canonical_headshot_url")
            .alias("headshot_url"),

        # lineage de la observación más reciente
        "source_entity",
        "source_file",
        "source_modified_at",
        "bronze_ingested_at"
    )
)

In [0]:
rescued_count = (
    driver_candidates_df
    .filter(col("_rescued_data").isNotNull())
    .count()
)

if rescued_count > 0:
    raise ValueError(
        f"Driver sources contain {rescued_count} records "
        "with _rescued_data."
    )

null_driver_id_count = (
    driver_candidates_df
    .filter(col("driver_id").isNull())
    .count()
)

if null_driver_id_count > 0:
    raise ValueError(
        f"Driver sources contain {null_driver_id_count} "
        "records without driver_id."
    )

abbreviation_conflicts = (
    driver_candidates_df
    .filter(col("abbreviation").isNotNull())
    .groupBy("driver_id")
    .agg(
        countDistinct("abbreviation")
            .alias("abbreviation_count")
    )
    .filter(col("abbreviation_count") > 1)
)

if abbreviation_conflicts.limit(1).count() > 0:
    raise ValueError(
        "One or more driver_id values are associated "
        "with multiple abbreviations."
    )

In [0]:
validation = (
    driver_candidates_df
    .agg(
        sum_spark(
            when(
                col("driver_id").isNull(),
                1
            ).otherwise(0)
        ).alias("null_driver_id"),

        sum_spark(
            when(
                col("abbreviation").isNull(),
                1
            ).otherwise(0)
        ).alias("null_abbreviation"),

        sum_spark(
            when(
                col("full_name").isNull(),
                1
            ).otherwise(0)
        ).alias("null_full_name"),

        sum_spark(
            when(
                col("first_name").isNull(),
                1
            ).otherwise(0)
        ).alias("null_first_name"),

        sum_spark(
            when(
                col("last_name").isNull(),
                1
            ).otherwise(0)
        ).alias("null_last_name")
    )
    .first()
    .asDict()
)

errors = {
    rule: value or 0
    for rule, value in validation.items()
    if (value or 0) > 0
}

duplicate_count = (
    driver_candidates_df
    .groupBy("driver_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

if duplicate_count > 0:
    errors["duplicate_driver_id"] = duplicate_count

if errors:
    raise ValueError(
        f"Silver drivers validation failed: {errors}"
    )

print(
    f"Validation OK: {driver_candidates_df.count()} drivers ready for Silver."
)

## Insercion - Races

In [0]:
if not spark.catalog.tableExists(SILVER_DRIVERS_TABLE):
    target = DeltaTable.forName(
        spark,
        SILVER_DRIVERS_TABLE
    )

    (
        target.alias("target")
        .merge(
            driver_candidates_df.alias("source"),
            """
            target.driver_id = source.driver_id
            """
        )
        .whenMatchedUpdateAll(
            condition="""
                target._record_hash <> source._record_hash
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged data into {SILVER_DRIVERS_TABLE}")

else:

    (
        driver_candidates_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_DRIVERS_TABLE)
    )

    print(f"Created {SILVER_DRIVERS_TABLE}")